### Context Handling and Memory (with Scratchpad or Session)

In this tutorial, we'll build agents that can remember previous conversations
and use a simple scratchpad to track their thinking. This makes agents much
more helpful for ongoing conversations.

Think of this as giving your agent a simple notebook to remember what you've
talked about and jot down its thinking process.

Learning Objectives:
- Build agents that remember previous conversations
- Create a simple scratchpad for agent reasoning
- Handle multi-turn conversations effectively
- Store and retrieve conversation context

In [1]:
# Install required packages
# pip install pydantic-ai openai python-dotenv

import os
from typing import List, Optional, Dict, Any
from datetime import datetime
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

True

### Simple Memory Structures

Let's start with simple structures for storing conversation memory.

In [2]:
class ConversationMemory(BaseModel):
    """
    Simple memory for storing conversation history.
    """
    topic: str = Field(description="What we're talking about")
    details: str = Field(description="Important details to remember")
    timestamp: datetime = Field(description="When this was discussed")

class ScratchpadNote(BaseModel):
    """
    A simple note in the agent's thinking process.
    """
    step: int = Field(description="Step number in the thinking process")
    thought: str = Field(description="What the agent is thinking")
    conclusion: str = Field(description="What the agent concluded")

class SimpleSession(BaseModel):
    """
    A simple session to track conversations.
    """
    session_id: str = Field(description="Unique session ID")
    memories: List[ConversationMemory] = Field(description="Things we've discussed")
    scratchpad: List[ScratchpadNote] = Field(description="Agent's thinking notes")
    conversation_summary: str = Field(description="Brief summary of our conversation")

### Simple Memory Manager

Now let's create a simple system to manage memory and conversations.

In [3]:
class SimpleMemoryManager:
    """
    A simple memory manager for our agent conversations.
    """
    
    def __init__(self):
        self.sessions: Dict[str, SimpleSession] = {}
        self.current_session_id = "demo_session"
        
        # Create a default session
        self.sessions[self.current_session_id] = SimpleSession(
            session_id=self.current_session_id,
            memories=[],
            scratchpad=[],
            conversation_summary="New conversation started"
        )
    
    def add_memory(self, topic: str, details: str):
        """Add something important to remember."""
        memory = ConversationMemory(
            topic=topic,
            details=details,
            timestamp=datetime.now()
        )
        self.sessions[self.current_session_id].memories.append(memory)
        print(f"💾 Remembered: {topic}")
    
    def add_scratchpad_note(self, step: int, thought: str, conclusion: str):
        """Add a note to the agent's thinking process."""
        note = ScratchpadNote(
            step=step,
            thought=thought,
            conclusion=conclusion
        )
        self.sessions[self.current_session_id].scratchpad.append(note)
        print(f"🧠 Thinking step {step}: {thought}")
    
    def get_memories_about(self, topic: str) -> List[ConversationMemory]:
        """Find memories related to a topic."""
        session = self.sessions[self.current_session_id]
        related_memories = []
        
        for memory in session.memories:
            if topic.lower() in memory.topic.lower() or topic.lower() in memory.details.lower():
                related_memories.append(memory)
        
        return related_memories
    
    def get_scratchpad(self) -> List[ScratchpadNote]:
        """Get all the agent's thinking notes."""
        return self.sessions[self.current_session_id].scratchpad
    
    def update_summary(self, new_summary: str):
        """Update the conversation summary."""
        self.sessions[self.current_session_id].conversation_summary = new_summary
        print(f"📝 Updated conversation summary")

### Creating a Memory-Aware Agent

Now let's create an agent that can use memory and a scratchpad.

In [4]:
memory_manager = SimpleMemoryManager()

In [5]:
class MemoryResponse(BaseModel):
    """
    Response from our memory-aware agent.
    """
    response: str = Field(description="The main response to the user")
    memories_used: List[str] = Field(description="What memories the agent referenced")
    new_memories: List[str] = Field(description="New things the agent learned")
    thinking_steps: List[str] = Field(description="Agent's reasoning process")


In [6]:
memory_agent = Agent(
    'openai:gpt-4o',
    output_type=MemoryResponse,
    system_prompt="""
    You are an AI agent with simple memory capabilities.
    
    Your approach:
    1. Check if you remember anything about the topic
    2. Use your scratchpad to think through the problem
    3. Provide a helpful response
    4. Remember important things for future conversations
    
    Be friendly and show that you're building understanding over time.
    """
)

In [7]:
@memory_agent.tool
async def check_memory(ctx: RunContext[None], topic: str) -> str:
    """
    Check what we remember about a topic.
    """
    memories = memory_manager.get_memories_about(topic)
    
    if not memories:
        return f"I don't have any memories about {topic} yet."
    
    memory_summary = f"I remember {len(memories)} things about {topic}:\n"
    for i, memory in enumerate(memories, 1):
        memory_summary += f"{i}. {memory.details}\n"
    
    return memory_summary

In [8]:
@memory_agent.tool
async def add_to_memory(ctx: RunContext[None], topic: str, details: str) -> str:
    """
    Remember something important.
    """
    memory_manager.add_memory(topic, details)
    return f"I'll remember this about {topic}: {details}"

In [9]:
@memory_agent.tool
async def think_step_by_step(ctx: RunContext[None], step: int, thought: str, conclusion: str) -> str:
    """
    Add a step to my thinking process.
    """
    memory_manager.add_scratchpad_note(step, thought, conclusion)
    return f"Thinking step {step} completed: {conclusion}"

In [10]:
@memory_agent.tool
async def review_thinking(ctx: RunContext[None]) -> str:
    """
    Review my thinking process so far.
    """
    notes = memory_manager.get_scratchpad()
    
    if not notes:
        return "I haven't written down any thoughts yet."
    
    thinking_summary = f"My thinking process ({len(notes)} steps):\n"
    for note in notes:
        thinking_summary += f"Step {note.step}: {note.thought} → {note.conclusion}\n"
    
    return thinking_summary

### Multi-Turn Conversation Demo

Let's see how our agent handles a conversation that builds over time.

In [11]:
# Conversation turns that build on each other
conversation = [
    "I'm thinking about starting a small business",
    "What are the key things I should consider for my business?",
    "How much money do I typically need to start?",
    "What about the business plan we discussed?"
]

In [12]:
for i, user_input in enumerate(conversation, 1):
    print(f"=== TURN {i} ===")
    print(f"You: {user_input}")
    
    try:
        result = await memory_agent.run(user_input)
        
        print(f"Agent: {result.output.response}")
        
        if result.output.memories_used:
            print(f"Used memories: {result.output.memories_used}")
        
        if result.output.new_memories:
            print(f"New memories: {result.output.new_memories}")
        
        if result.output.thinking_steps:
            print(f"Thinking: {result.output.thinking_steps}")
            
    except Exception as e:
        print(f"Error: {e}")
    
    print()

=== TURN 1 ===
You: I'm thinking about starting a small business
🧠 Thinking step 1: The user is interested in starting a small business. I should guide them through the initial steps involved and considerations they should keep in mind.
Agent: Starting a small business is an exciting endeavor! Here are some key steps to get you started:

1. **Identify Your Business Idea**: Think about what you're passionate about or areas where you have experience. Consider trends in the market or gaps that you could fill with a unique service or product.

2. **Conduct Market Research**: Understand your target audience, competition, and demand for your product or service.

3. **Create a Business Plan**: This will be your roadmap. It should outline your business goals, strategies, financial projections, and operational plans.

4. **Legal Considerations**: Decide the legal structure of your business (e.g., sole proprietorship, LLC, corporation) and register your business name. Also, look into the permits